# CP8305 Final Project
## Notebook 2 – Data Preprocessing

**Team members:** Christopher Indris; Mojtaba Mohammadi; Mohammad Al Batayneh  
**Date:** 2026-03-09

---

### Instructions
Run with a fresh 3.11 environment (I used a 3.11.11 venv "cp8305fp").
Ensure that you pip install ipykernel to run the notebook.

---

### Objectives
- Remove duplicates and handle missing values.
- Detect and treat outliers.
- Encode categorical variables.
- Engineer new features using domain knowledge.
- Scale numerical features.
- Save the processed dataset for modelling.

**Portfolio maintenance:** paths now resolve from the repository root. Original figures, tables and submitted documents are retained separately. See `docs/VALIDATION.md` for exactly which revised notebooks were executed.

## 1. Setup

In [ ]:
# Install dependencies before starting Jupyter; see README.md.

In [ ]:
# Resolve imports from either the repository root or notebooks/.
import sys
from pathlib import Path
PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / 'src' / 'paths.py').exists() and (p / 'notebooks').is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT))
from src.paths import RAW_DATA_DIR, PROCESSED_DATA_DIR, TABLES_DIR
TABLES_DIR.mkdir(parents=True, exist_ok=True)

import io
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_preprocessing import (
    load_raw_data,
    drop_duplicate_rows,
    handle_missing_values,
    remove_outliers_iqr,
    encode_categoricals,
    encode_categoricals_wrapper,
    save_processed_data,
)
from src.feature_engineering import (
    PREPROCESSING_RULES,
    preprocess_numericals_for_model,
    scale_features,
    add_interaction_term,
    add_log_transform,
    add_binned_column,
)
from src.visualization import plot_count, plot_missing_values, save_figure

# Fix random seed to keep preprocessing outputs reproducible across runs.
np.random.seed(42)

sns.set_theme(style='whitegrid')
%matplotlib inline

## 2. Load Data

In [ ]:
data_source = 'local'

In [ ]:
# Original UCI files downloaded with: python -m src.download_data
DATA_FILE = RAW_DATA_DIR / 'diabetic_data.csv'
if not DATA_FILE.exists():
    raise FileNotFoundError('Download the dataset from the repository root: python -m src.download_data')
df = pd.read_csv(DATA_FILE, keep_default_na=False, na_values=['?'], low_memory=False)
idXy = df.copy()
X = df.drop(columns=['encounter_id', 'patient_nbr', 'readmitted']).copy()
y = df[['readmitted']].copy()

### Mappings for encoded values

In [ ]:
def read_multiple_tables_csv(filepath, table_delimiter=',\n'):
    """
    Reads a single CSV file containing multiple tables into a list of DataFrames.
    """
    with open(filepath, "r") as f:
        csv_content = f.read()
    
    # Split the content into a list of strings, one for each table
    csv_tables_list = csv_content.split(table_delimiter)
    
    # Convert each string segment into a DataFrame
    dataframe_list = [pd.read_csv(io.StringIO(table_data)) for table_data in csv_tables_list if table_data.strip()]
    
    return dataframe_list

In [ ]:
admission_type_ids, discharge_disposition_ids, admission_source_id = read_multiple_tables_csv(RAW_DATA_DIR / 'IDS_mapping.csv')

## 3. Remove Duplicates

There are no duplicates to drop.

In [ ]:
int(X.duplicated().sum()), int(idXy.duplicated().sum())

## 4. Missing Values

### Ordinal Categorical

- weight
Since weight is ordinal and categorical (weight is numeric, and it was mapped to ranges of 25 lbs.), we could estimate it by taking the mode (a kind of analogue to the mean). \
However, there may be correlations between it and other variables; in particular, age (which is ordinal categorical from a numeric, mapped to ranges of 10 years). Age also has no missing values. Hence, we could try setting for each row where weight is missing the weight equal to the mode of the weight of all rows where the age is the same as that row.

The problem is that since almost all of the weight column is missing, we can't estimate it.

NOTE: existing papers might drop the weight as well.

In [ ]:
modes = X['weight'].groupby(X['age']).agg(pd.Series.mode).reset_index(drop=False)
modes['weight'] = modes['weight'].apply(lambda x: x if isinstance(x, str) else x[0])
modes

In [ ]:
modes.loc[modes['age'] == '[0-10)', 'weight'].values[0]

However, some of them like A1Cresult and max_glu_serum, it is unclear what we would fill those values with. Also, we shouldn't need to make them explicitly ordinal; our ML functions will decide on how each of the values correlate with the output (especially since there are only a few different classes for each of A1Cresult and max_glu_serum)

In [ ]:
# calculate which columns have missing values; we will take the union of those listed in the UCI metadata and those calculated here, so that we don't miss any

# computed directly
mvs = idXy.isna().sum().reset_index(name='missing_values')
mvs = mvs[mvs['missing_values'] > 0]['index'].tolist()

# using metadata
if data_source == 'uciml':
    mvs_meta = diabetes_130_us_hospitals_for_years_1999_2008.variables[diabetes_130_us_hospitals_for_years_1999_2008.variables.missing_values == 'yes'].name.tolist()
else:
    mvs_meta = [] # would pick up A1Cresult and max_glu_serum

columns_with_missing_values = list(set(mvs).union(set(mvs_meta)))
columns_with_missing_values

In [ ]:
idXy[columns_with_missing_values].isna().sum()

In [ ]:
# the first 5 rows all have missing weight
idXy[['encounter_id', 'patient_nbr', 'weight', 'age']].head()

In [ ]:
def replace_missing_values(df, columns, replace_with):
    if isinstance(columns, tuple):
        columns, helper_columns = columns
    if replace_with == 'mode':
        for col in columns:
            df[col] = df[col].fillna(df[col].mode().values[0], inplace=False)
    elif replace_with == 'mode_based_on_helper_column':
        for col in columns:
            modes = df[col].groupby(df[helper_columns[0]]).agg(pd.Series.mode).reset_index(drop=False)
            modes[col] = modes[col].apply(lambda x: x if isinstance(x, str) else x[0])
            df[col] = df.apply(lambda row: modes.loc[modes[helper_columns[0]] == row[helper_columns[0]], col].values[0] if pd.isna(row[col]) else row[col], axis=1)
    return df

In [ ]:
idXy = replace_missing_values(idXy, (['weight'], ['age']), 'mode_based_on_helper_column')
X = replace_missing_values(X, (['weight'], ['age']), 'mode_based_on_helper_column')

In [ ]:
idXy['weight'].value_counts(dropna=False)

In [ ]:
# the weight have been fixed according to our mapping
idXy[['encounter_id', 'patient_nbr', 'weight', 'age']].head()

In [ ]:
columns_with_missing_values.remove('weight')

In [ ]:
# nearly all are missing
idXy['weight'].value_counts(dropna=False)

NOTE: the "papers" on Google Drive drop the weight column because it is too sparse. 

In [ ]:
# drop weight from idXy
idXy = idXy.drop(columns=['weight'])
# drop weight from X
X = X.drop(columns=['weight'])

### Nominal Categorical

These are categorical values where we cannot estimate the missing value, so we can simply say "NA".

In [ ]:
idXy[columns_with_missing_values].info()

In [ ]:
if data_source == 'uciml':
    diabetes_130_us_hospitals_for_years_1999_2008.variables[diabetes_130_us_hospitals_for_years_1999_2008.variables.name.isin(columns_with_missing_values)].values

array([['race', 'Feature', 'Categorical', 'Race',
        'Values: Caucasian, Asian, African American, Hispanic, and other',
        None, 'yes'],
       ['payer_code', 'Feature', 'Categorical', nan,
        'Integer identifier corresponding to 23 distinct values, for example, Blue Cross/Blue Shield, Medicare, and self-pay',
        None, 'yes'],
       ['medical_specialty', 'Feature', 'Categorical', nan,
        'Integer identifier of a specialty of the admitting physician, corresponding to 84 distinct values, for example, cardiology, internal medicine, family/general practice, and surgeon',
        None, 'yes'],
       ['diag_1', 'Feature', 'Categorical', nan,
        'The primary diagnosis (coded as first three digits of ICD9); 848 distinct values',
        None, 'yes'],
       ['diag_2', 'Feature', 'Categorical', nan,
        'Secondary diagnosis (coded as first three digits of ICD9); 923 distinct values',
        None, 'yes'],
       ['diag_3', 'Feature', 'Categorical', nan,
        'Additional secondary diagnosis (coded as first three digits of ICD9); 954 distinct values',
        None, 'yes'],
       ['max_glu_serum', 'Feature', 'Categorical', nan,
        'Indicates the range of the result or if the test was not taken. Values: >200, >300, normal, and none if not measured',
        None, 'no'],
       ['A1Cresult', 'Feature', 'Categorical', nan,
        'Indicates the range of the result or if the test was not taken. Values: >8 if the result was greater than 8%, >7 if the result was greater than 7% but less than 8%, normal if the result was less than 7%, and none if not measured.',
        None, 'no']], dtype=object)

#### Race

In [ ]:
idXy['race'].value_counts(dropna=False)

In [ ]:
# for column 'race' of idXy, replace nan with string "NA"
idXy['race'] = idXy['race'].fillna('NA')
# for column 'race' of X, replace nan with string "NA"
X['race'] = X['race'].fillna('NA')
idXy['race'].value_counts(dropna=False)


In [ ]:
columns_with_missing_values.remove('race')

#### Payer Code

NOTE: it seems that lots of papers (the papers in our Google drive) drop this column due to lack of values.

In [ ]:
idXy['payer_code'].value_counts(dropna=False)

In [ ]:
# for column 'payer_code' of idXy, replace nan with string "NA"
idXy['payer_code'] = idXy['payer_code'].fillna('NA')
# for column 'payer_code' of X, replace nan with string "NA"
X['payer_code'] = X['payer_code'].fillna('NA')
idXy['payer_code'].value_counts(dropna=False)

In [ ]:
columns_with_missing_values.remove('payer_code')

In [ ]:
# completely drop the payer_code column from idXy and X
idXy = idXy.drop(columns=['payer_code'])
X = X.drop(columns=['payer_code'])


#### Medical Specialty

NOTE: it seems that papers 5 and 7 drop these. Recent papers can either drop, or re-cardinalize. There are also many classes (73). Hence, perhaps we can drop.

In [ ]:
idXy['medical_specialty'].value_counts(dropna=False)

In [ ]:
# for column 'medical_specialty' of idXy, replace nan with string "NA"
idXy['medical_specialty'] = idXy['medical_specialty'].fillna('NA')
# for column 'medical_specialty' of X, replace nan with string "NA"
X['medical_specialty'] = X['medical_specialty'].fillna('NA')
idXy['medical_specialty'].value_counts(dropna=False)

In [ ]:
# completely drop the medical_specialty column from idXy and X
idXy = idXy.drop(columns=['medical_specialty'])
X = X.drop(columns=['medical_specialty'])

#### Diagnoses (1, 2, 3)

In [ ]:
columns_with_missing_values.remove('medical_specialty')

In [ ]:
len(idXy[idXy['diag_1'].isna()]), len(idXy[idXy['diag_2'].isna()]), len(idXy[idXy['diag_3'].isna()])

In [ ]:
idXy['diag_1'] = idXy['diag_1'].fillna('NA')
X['diag_1'] = X['diag_1'].fillna('NA')
idXy['diag_2'] = idXy['diag_2'].fillna('NA')
X['diag_2'] = X['diag_2'].fillna('NA')
idXy['diag_3'] = idXy['diag_3'].fillna('NA')
X['diag_3'] = X['diag_3'].fillna('NA')

columns_with_missing_values.remove('diag_1')
columns_with_missing_values.remove('diag_2')
columns_with_missing_values.remove('diag_3')


In [ ]:
len(idXy[idXy['diag_1'].isna()]), len(idXy[idXy['diag_2'].isna()]), len(idXy[idXy['diag_3'].isna()])

In [ ]:
columns_with_missing_values

#### max_glu_serum

Note that in the CSV file, this is "None", not "NaN". This is because there is a difference in medical tests between "not ordered" and "not available". Hence, we can keep this in for now. (Paper 1 did not drop this column).

When we open the .csv in Excel, we see "?" for missing values and "None" as a proper category.

Hence, for this column I will use "None" rather than "NA".

In [ ]:
idXy['max_glu_serum'].value_counts(dropna=False)

In [ ]:
# for column 'max_glu_serum' of idXy, replace nan with string "None" since it is a proper category
idXy['max_glu_serum'] = idXy['max_glu_serum'].fillna('None')
# for column 'max_glu_serum' of X, replace nan with string "None" since it is a proper category
X['max_glu_serum'] = X['max_glu_serum'].fillna('None')
idXy['max_glu_serum'].value_counts(dropna=False)

In [ ]:
try:
    columns_with_missing_values.remove('max_glu_serum')
except:
    pass

### A1Cresult

Note that in the CSV file, this is "None", not "NaN". This is because there is a difference in medical tests between "not ordered" and "not available". Hence, we can keep this in for now. (Paper 1 did not drop this column).

When we open the .csv in Excel, we see "?" for missing values and "None" as a proper category.

Hence, for this column I will use "None" rather than "NA".

In [ ]:
idXy['A1Cresult'].value_counts(dropna=False)

In [ ]:
# for column 'A1Cresult' of idXy, replace nan with string "None" since it is a proper category
idXy['A1Cresult'] = idXy['A1Cresult'].fillna('None')
# for column 'A1Cresult' of X, replace nan with string "None" since it is a proper category
X['A1Cresult'] = X['A1Cresult'].fillna('None')
idXy['max_glu_serum'].value_counts(dropna=False)

In [ ]:
try:
    columns_with_missing_values.remove('A1Cresult')
except:
    pass

## 4. Modifying Data Types

In some cases, even though the data is integer (numerical ordinal), it should be treated as categorical. Examples:

- admission_type_id
- discharge_disposition_id
- admission_source_id

In [ ]:
idXy.info()

In [ ]:
# convert the datatype of columns 'admission_type_id', 'discharge_disposition_id', 'admission_source_id' to categorical
idXy['admission_type_id'] = idXy['admission_type_id'].astype('category')
idXy['discharge_disposition_id'] = idXy['discharge_disposition_id'].astype('category')
idXy['admission_source_id'] = idXy['admission_source_id'].astype('category')

X['admission_type_id'] = X['admission_type_id'].astype('category')
X['discharge_disposition_id'] = X['discharge_disposition_id'].astype('category')
X['admission_source_id'] = X['admission_source_id'].astype('category')

idXy.info()


In [ ]:
# Convert all string-type columns in idXy and X to category dtype
string_columns_idXy = idXy.select_dtypes(include=['object']).columns
for col in string_columns_idXy:
    idXy[col] = idXy[col].astype('category')
    if col in X.columns:
        X[col] = X[col].astype('category')

In [ ]:
idXy.info()

In [ ]:
# Let's convert our IDs, 'encounter_id' and 'patient_nbr', to categorical.
idXy['encounter_id'] = idXy['encounter_id'].astype('category')
idXy['patient_nbr'] = idXy['patient_nbr'].astype('category')
idXy.info()


In [ ]:
idXy['age'].value_counts(dropna=False)

age

We could add a column called "age_numeric" where [0-10) -> 0, [10-20)->1, etc. This is in the style of paper 1.

## 5. Outlier Treatment and Numeric Attribute Normalization

### Finding Outliers

In [ ]:
# find the columns which are numeric (type int64); the IDs have already been converted to categorical ('encounter_id' or 'patient_nbr')
numerical_nonID_columns = idXy.select_dtypes(include=['int64'])
numerical_nonID_columns


In [ ]:
numerical_nonID_columns = numerical_nonID_columns.columns

In [ ]:
#idXy[numerical_nonID_columns].describe(include='all').T
idXy[numerical_nonID_columns].describe(include='all').T

In [ ]:
idXy[numerical_nonID_columns].hist(figsize=(12, 10), layout=(3, 3), sharex=False, sharey=False)
plt.tight_layout(pad=3.0)

In [ ]:
idXy[numerical_nonID_columns].boxplot(figsize=(12, 7), boxprops=dict(lw=2))
plt.tight_layout(pad=3.0)

In [ ]:
# NOTE: we won't get rid of the outliers.

# Calculate outlier counts per numeric column based on the IQR fence.
outlier_counts = []
total_rows = idXy.shape[0]
for col in numerical_nonID_columns:
    q1 = idXy[col].quantile(0.25)
    q3 = idXy[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = idXy[(idXy[col] < lower) | (idXy[col] > upper)]
    count = outliers.shape[0]
    percent = (count / total_rows) * 100 if total_rows > 0 else 0
    outlier_counts.append({
        'column': col,
        'outlier_count': count,
        'outlier_percent': percent,
    })

outlier_counts_df = pd.DataFrame(outlier_counts).sort_values(
    by='outlier_percent', ascending=False
)
print('Number and percentage of outliers per numeric column (IQR rule):')
display(outlier_counts_df)

# Outliers are described, not removed: high utilization can be meaningful.
print(f'Rows retained (no IQR filtering applied): {len(idXy):,}')

### Adjusting the Numerical Scales

In [ ]:
rules_df = pd.DataFrame(PREPROCESSING_RULES).T
rules_df.index.name = "column"
print("Model-specific preprocessing rules for numerical columns:")
display(rules_df)

In [ ]:
model_types = ["knn", "decision_tree", "logistic_regression", "gaussian_nb"]
cols = list(PREPROCESSING_RULES.keys())

for model_type in model_types:
    transformed_df, fitted = preprocess_numericals_for_model(idXy, model_type)

    before_stats = idXy[cols].describe().loc[["mean", "std", "min", "max"]].T
    after_stats = transformed_df[cols].describe().loc[["mean", "std", "min", "max"]].T

    summary = before_stats.join(after_stats, lsuffix="_before", rsuffix="_after")
    summary.columns = [
        "mean_before", "std_before", "min_before", "max_before",
        "mean_after", "std_after", "min_after", "max_after",
    ]

    transforms_used = {
        c: PREPROCESSING_RULES[c].get(model_type, "none") for c in cols
    }
    summary.insert(0, "transform", [transforms_used[c] for c in summary.index])

    print(f"\n{'=' * 80}")
    print(f"Model: {model_type}")
    print(f"{'=' * 80}")
    display(summary.round(4))

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes_flat = axes.ravel()

model_type = "knn"
transformed_df, _ = preprocess_numericals_for_model(idXy, model_type)

for i, col in enumerate(cols):
    ax = axes_flat[i]
    transform_name = PREPROCESSING_RULES[col][model_type]
    ax.hist(idXy[col], bins=50, alpha=0.5, label="Original", density=True)
    ax.hist(
        transformed_df[col], bins=50, alpha=0.5,
        label=f"After ({transform_name})", density=True,
    )
    ax.set_title(col)
    ax.legend(fontsize=8)

fig.suptitle(
    f"Distributions: Original vs. Transformed (model={model_type})",
    fontsize=14, y=1.01,
)
plt.tight_layout()
plt.show()

## 6. Feature Engineering

### 6.1 Diagnosis Consolidation

In [ ]:
# Strack-style ICD-9 collapse to 9 high-level categories used in the literature.
def icd9_to_strack_group(code):
    if pd.isna(code):
        return "Other"

    code_str = str(code).strip()
    if code_str in {"", "?", "nan", "NA"}:
        return "Other"

    code_upper = code_str.upper()
    if code_upper.startswith(("V", "E")):
        return "Other"

    try:
        value = float(code_str)
    except ValueError:
        return "Other"

    if int(value) == 250:
        return "Diabetes"
    if 390 <= value <= 459 or int(value) == 785:
        return "Circulatory"
    if 460 <= value <= 519 or int(value) == 786:
        return "Respiratory"
    if 520 <= value <= 579 or int(value) == 787:
        return "Digestive"
    if 800 <= value <= 999:
        return "Injury"
    if 710 <= value <= 739:
        return "Musculoskeletal"
    if 580 <= value <= 629 or int(value) == 788:
        return "Genitourinary"
    if 140 <= value <= 239:
        return "Neoplasms"
    return "Other"

In [ ]:
for diag_col in ["diag_1", "diag_2", "diag_3"]:
    group_col = f"{diag_col}_group"
    # Keep grouped diagnoses on both frames to support diagnostics and feature engineering.
    idXy[group_col] = idXy[diag_col].apply(icd9_to_strack_group).astype("category")
    X[group_col] = X[diag_col].apply(icd9_to_strack_group).astype("category")

In [ ]:
fig = plot_count(idXy, ['diag_1_group', 'diag_2_group', 'diag_3_group'], 1)
save_figure(fig, '01_counts_diagnosis_grouped.png')
plt.show()

In [ ]:
# Incorporate KNN rule transforms into idXy before section 6.2 so df = idXy.copy() and
# processed_dataset.csv match the "After" distributions in the figure above.
# NUMERICAL_PREPROCESS_MODEL = "knn"
# idXy, _ = preprocess_numericals_for_model(idXy, NUMERICAL_PREPROCESS_MODEL)


### 6.2 Additional Computed Categories

Collinearity notes for engineered features:
- both_abnormal is derived from a1c_abnormal and glu_abnormal, so it is intentionally redundant.
- total_prior_visits overlaps with number_inpatient, number_emergency, and number_outpatient.
- comorbidity_count summarizes diag_1_group, diag_2_group, and diag_3_group and should be interpreted as an aggregate acuity signal.

In [ ]:
# Build modeling dataset; outliers were described but retained.
df = idXy.copy()

# Exclude expired/hospice discharge codes for this study cohort.
terminal_or_hospice_ids = {'11', '13', '14', '19', '20', '21'}
rows_before_terminal_filter = len(df)
df = df[
    ~df['discharge_disposition_id'].astype(str).str.strip().isin(terminal_or_hospice_ids)
]
print(
    'Rows removed (terminal/hospice outcomes): '
    f"{rows_before_terminal_filter - len(df)}"
)

# Keep one eligible encounter per patient, ordered by encounter_id (not a verified timestamp).
rows_before_patient_dedup = len(df)
encounter_sort_key = pd.to_numeric(df['encounter_id'], errors='coerce')
df = (
    df.assign(_encounter_sort_key=encounter_sort_key)
      .sort_values('_encounter_sort_key', kind='stable')
      .drop_duplicates(subset=['patient_nbr'], keep='first')
      .drop(columns=['_encounter_sort_key'])
      .copy()
 )
print(
    'Rows removed (repeat patient encounters): '
    f"{rows_before_patient_dedup - len(df)}"
)

# Binary target: 1 if readmitted in <30 days, else 0.
df['readmitted_binary'] = (df['readmitted'].astype(str) == '<30').astype('int8')

# Remove original multiclass target to avoid leakage/confusion.
df = df.drop(columns=['readmitted'])

# Decode clinical IDs using IDS_mapping lookup tables loaded earlier.
def normalize_id(value):
    text = str(value).strip()
    try:
        return str(int(float(text)))
    except ValueError:
        return text


def build_lookup(mapping_df, id_col):
    cleaned = mapping_df[[id_col, 'description']].dropna(subset=[id_col]).copy()
    cleaned[id_col] = cleaned[id_col].apply(normalize_id)
    cleaned['description'] = cleaned['description'].astype(str).str.strip()
    return dict(zip(cleaned[id_col], cleaned['description']))


admission_lookup = build_lookup(admission_type_ids, 'admission_type_id')
discharge_lookup = build_lookup(discharge_disposition_ids, 'discharge_disposition_id')
source_lookup = build_lookup(admission_source_id, 'admission_source_id')

df['admission_type_desc'] = (
    df['admission_type_id'].astype(object).apply(normalize_id).map(admission_lookup).fillna('Unknown')
)
df['discharge_disposition_desc'] = (
    df['discharge_disposition_id'].astype(object).apply(normalize_id).map(discharge_lookup).fillna('Unknown')
)
df['admission_source_desc'] = (
    df['admission_source_id'].astype(object).apply(normalize_id).map(source_lookup).fillna('Unknown')
)

# Encode age brackets as ordinal values rather than continuous midpoints.
age_order = [
    '[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)',
    '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)',
]
age_to_ordinal = {label: idx + 1 for idx, label in enumerate(age_order)}
df['age_ordinal'] = df['age'].astype(str).map(age_to_ordinal).fillna(0).astype('int8')

# -------------------------
# Utilization intensity
# -------------------------
# Aggregate prior utilization into one acuity-oriented signal.
df['total_prior_visits'] = (
    df['number_outpatient'] + df['number_emergency'] + df['number_inpatient']
)

stay_days = df['time_in_hospital'].replace(0, np.nan)
df['labs_per_day'] = (df['num_lab_procedures'] / stay_days).fillna(0.0)
df['meds_per_day'] = (df['num_medications'] / stay_days).fillna(0.0)

# -------------------------
# Medication dynamics
# -------------------------
medication_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide',
    'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone',
]
available_meds = [c for c in medication_cols if c in df.columns]

df['num_active_meds'] = (
    (df[available_meds].astype(str) != 'No').sum(axis=1)
).astype('int16')

# Count all dosage changes across the diabetes medication list.
df['total_drug_changes'] = (
    df[available_meds].astype(str).isin(['Up', 'Down']).sum(axis=1)
).astype('int16')

df['any_diabetes_med_change'] = (df['change'].astype(str) == 'Ch').astype('int8')
df['insulin_changed'] = df['insulin'].astype(str).isin(['Up', 'Down']).astype('int8')

# -------------------------
# Consolidate lab risk categories
# -------------------------
# Keep row-wise indicators (fixed: no value_counts misuse).
df['a1c_abnormal'] = df['A1Cresult'].astype(str).isin(['>7', '>8']).astype('int8')
df['glu_abnormal'] = df['max_glu_serum'].astype(str).isin(['>200', '>300']).astype('int8')
df['both_abnormal'] = ((df['a1c_abnormal'] == 1) & (df['glu_abnormal'] == 1)).astype('int8')

# -------------------------
# Grouped admission/discharge/source features
# -------------------------
admission_groups = {
    'emergency_or_trauma': {'1', '7'},
    'urgent': {'2'},
    'elective': {'3'},
    'newborn': {'4'},
}

discharge_groups = {
    'home': {'1', '6', '8'},
    'transfer': {'2', '3', '4', '5', '10', '15', '22', '23', '24', '27', '28', '29', '30'},
    'expired': {'11', '19', '20', '21'},
    'hospice': {'13', '14'},
    'left_ama': {'7'},
    'inpatient': {'9'},
}

source_groups = {
    'er': {'7'},
    'referral': {'1', '2', '3'},
    'transfer': {'4', '5', '6', '10', '18', '22', '25', '26'},
    'newborn': {'11', '12', '13', '14', '23', '24'},
    'home_health_readmit': {'19'},
}


def map_group(code, groups, default='other'):
    code = str(code).strip()
    for label, codes in groups.items():
        if code in codes:
            return label
    return default


df['admission_type_group'] = df['admission_type_id'].astype(str).apply(
    lambda x: map_group(x, admission_groups)
).astype('category')
df['discharge_group'] = df['discharge_disposition_id'].astype(str).apply(
    lambda x: map_group(x, discharge_groups)
).astype('category')
df['admission_source_group'] = df['admission_source_id'].astype(str).apply(
    lambda x: map_group(x, source_groups)
).astype('category')

# Summarize diagnosis burden by counting unique mapped systems per patient.
diag_group_cols = [c for c in ['diag_1_group', 'diag_2_group', 'diag_3_group'] if c in df.columns]
if diag_group_cols:
    def count_unique_non_other(values):
        cleaned = {
            v for v in values
            if v not in {'Other', 'NA', 'nan', 'None'}
        }
        return len(cleaned)

    df['comorbidity_count'] = (
        df[diag_group_cols]
        .astype(str)
        .apply(lambda row: count_unique_non_other(row), axis=1)
        .astype('int8')
    )
else:
    df['comorbidity_count'] = 0

print('Feature engineering complete.')
print(f'Columns after engineering: {df.shape[1]}')
print('readmitted_binary distribution:')
print(df['readmitted_binary'].value_counts(dropna=False).sort_index())


In [ ]:
idXy['A1Cresult'].value_counts(dropna=False)

## 7. Encode Categorical Variables

In [ ]:
# Drop leakage-prone identifiers before encoding/modeling.
id_columns = ['encounter_id', 'patient_nbr']
df = df.drop(columns=[c for c in id_columns if c in df.columns], errors='ignore')

# One-hot encode all remaining categorical predictors except the target.
df = encode_categoricals_wrapper(df, 'readmitted_binary', drop_first=True)

print('Dropped identifier columns before encoding:', id_columns)

In [ ]:
# Sanity checks for leakage and engineered features.
leakage_columns_remaining = [c for c in ['encounter_id', 'patient_nbr'] if c in df.columns]
print('Leakage columns still present:', leakage_columns_remaining)

required_engineered = [
    'total_prior_visits',
    'total_drug_changes',
    'a1c_abnormal',
    'glu_abnormal',
    'both_abnormal',
    'comorbidity_count',
]
missing_engineered = [c for c in required_engineered if c not in df.columns]
print('Missing engineered columns:', missing_engineered)

## 8. Scale Numeric Features

In [ ]:
# Keep scaling in the modeling pipeline to avoid leakage and double scaling.
target_column = 'readmitted_binary'
scale_cols = [
    c for c in df.select_dtypes(include=np.number).columns
    if c != target_column
]

# Intentionally disabled here: scaling must be fit on training folds only in Notebook 3.
# df, scaler = scale_features(df, scale_cols, method='standard')

print(f'Numeric columns available for scaling in modeling: {len(scale_cols)}')
print('Scaling is deferred to train-only fitting in Notebook 3.')


## 9. Final Check & Save

In [ ]:
print(f'Final shape  : {df.shape}')
print(f'Dtypes:\n{df.dtypes.value_counts()}')
print(f'Missing total: {df.isnull().sum().sum()}')
df.head()

In [ ]:
save_processed_data(df, 'processed_dataset.csv')
print('Preprocessing complete. Proceed to Notebook 3 – Modeling.')